# Análise de variantes somáticas com GATK Mutect2

Aula prática com um recorte da região de **JAK2**, usando amostras tumor–normal, GRCh37/hg19 e recursos preparados para fins didáticos.

> **Atenção:** este notebook é educacional e não constitui um pipeline clínico validado. Os nomes de contig do BAM, FASTA e VCF devem ser compatíveis.

## Objetivos

- preparar e indexar uma referência FASTA;
- executar Mutect2 em modo tumor–normal;
- estimar contaminação entre amostras;
- filtrar e inspecionar as variantes somáticas.

## Clonar Repositório 
> github.com/renatopuga/somatico

In [ ]:
!git clone https://github.com/renatopuga/somatico.git

## Preparando a Referência - chr9
Os arquivos BAM que vamos utilizar não contêm o prefixo `chr` no cabeçalho. Ex.: `>9`.
O arquivo que vamos fazer download `chr9.fa.gz` tem o cabeçalho `>chr9`, então precisaremos alterar essa parte para não termos erro de contigs quando rodar o GATK. Leia mais sobre  [*missing or incompatible contigs*](https://gatk.broadinstitute.org/hc/en-us/articles/360035891131-Errors-about-input-files-having-missing-or-incompatible-contigs)

### Download chr9.fa.gz
> `wget -c` faz download e resume o arquivo se necessário.

In [ ]:
!wget -c https://hgdownload.soe.ucsc.edu/goldenPath/hg19/chromosomes/chr9.fa.gz

### Alterar nome do cabeçalho do arquivo FASTA
> DE: `>chr9` para `>9`

In [ ]:
!zcat chr9.fa.gz | sed -e "s/chr//g" > chr9.fa

### Verificar a Mudança: comando `head`
> `head` lê as dez primeiras linhas de um arquivo

In [ ]:
!head chr9.fa

## samtools
Samtools é um conjunto de programas para interagir com dados de sequenciamento de alto rendimento.

### Instalar samtools

In [ ]:
!sudo apt-get install samtools 

### samtools faidx
> index/extract FASTA
Input é o arquivo `chr9.fa` e o saída o arquivo `chr9.fa.fai`.

In [ ]:
!samtools faidx chr9.fa

## Genome Analysis Toolkit - GATK4
> Versão: 4.6.2.0

Genome Analysis Toolkit - Variant Discovery in High-Throughput Sequencing Data. https://gatk.broadinstitute.org/

### Download

In [ ]:
!wget -c https://github.com/broadinstitute/gatk/releases/download/4.6.2.0/gatk-4.6.2.0.zip

### Descompactar
> comando `unzip`

In [ ]:
!unzip gatk-4.6.2.0.zip

### Testar o gatk
> se estiver ok aparecerá as opções do help do gatk

In [ ]:
!./gatk-4.6.2.0/gatk

## GATK4 - Criar arquivo .dict
> dicionário da referência para o GATK4


* CreateSequenceDictionary

Cria um dicionário de sequência para uma sequência de referência. Esta ferramenta cria um arquivo de dicionário de sequência (com extensão ".dict") a partir de uma sequência de referência fornecida no formato FASTA, que é exigido por muitas ferramentas de processamento e análise. O arquivo de saída contém um cabeçalho, mas nenhum SAMRecords, e o cabeçalho contém apenas registros de sequência.

* ScatterIntervalsByNs

Grava uma lista de intervalos criada pela divisão por Ns de uma referência.

### CreateSequenceDictionary
> Tool returned: 0

In [ ]:
!./gatk-4.6.2.0/gatk CreateSequenceDictionary -R chr9.fa -O chr9.dict

### ScatterIntervalsByNs

> Tool returned: 0



In [ ]:
!./gatk-4.6.2.0/gatk ScatterIntervalsByNs -R chr9.fa -O chr9.interval_list -OT ACGT

## Mutect2
Call somatic SNVs and indels via local assembly of haplotypes.

### Mutect2 - Tumor e Normal
> Tool returned: SUCCESS

In [ ]:
!./gatk-4.6.2.0/gatk Mutect2 \
  -R chr9.fa \
  -I somatico/tumor_JAK2.bam \
  -I somatico/normal_JAK2.bam \
  -normal WP044 \
  --germline-resource somatico/af-only-gnomad-chr9.vcf.gz \
  -O somatic.vcf.gz \
  -L chr9.interval_list

### GetPileupSummaries
Resume contagens de leituras que suportam referência, alelos alternativos e outros para determinados sites. Os resultados podem ser usados com CalculateContamination.


### GetPileupSummaries  - Amostra Tumor
> Tool returned: SUCCESS

In [ ]:
!./gatk-4.6.2.0/gatk GetPileupSummaries \
  -R chr9.fa \
  -I somatico/tumor_JAK2.bam \
  -V somatico/af-only-gnomad-chr9.vcf.gz \
  -L chr9.interval_list \
  -O tumor_JAK2.table

### GetPileupSummaries - Amostra Normal
> Tool returned: SUCCESS

In [ ]:
!./gatk-4.6.2.0/gatk GetPileupSummaries \
  -R chr9.fa \
  -I somatico/normal_JAK2.bam \
  -V somatico/af-only-gnomad-chr9.vcf.gz \
  -L chr9.interval_list \
  -O normal_JAK2.table

### CalculateContamination
Calcule a fração de reads provenientes da contaminação de amostra cruzada.

### CalculateContamination - Table
> Tool returned: SUCCESS

In [ ]:
!./gatk-4.6.2.0/gatk CalculateContamination \
  -I tumor_JAK2.table \
  -matched normal_JAK2.table \
  -O contamination.table

### FilterMutectCalls 
Filtrar chamada de SNVs e InDels somáticos chamados pelo Mutect2.

### FilterMutectCalls - Run

In [ ]:
!./gatk-4.6.2.0/gatk FilterMutectCalls \
  -R chr9.fa \
  -V somatic.vcf.gz \
  --contamination-table contamination.table \
  -O filtered.vcf.gz

## VEP ensembl
> release 116

### Instalar o VEP

In [ ]:
%%bash
set -euo pipefail
sudo apt-get update -qq
sudo apt-get install -y unzip curl git libmodule-build-perl libdbi-perl libdbd-mysql-perl build-essential zlib1g-dev
git clone --branch release/116 --depth 1 https://github.com/Ensembl/ensembl-vep.git
cd ensembl-vep
printf 'n\nn\nn\n' | perl INSTALL.pl

### VEP - Run
Consulte sobre as opções utilizadas no comando VEP abaixo. Leia sobre `--tab` e `--fields` para especificar a ordem da saída da tabela anotada: [--tab](https://www.ensembl.org/info/docs/tools/vep/vep_formats.html#tab) e [Todos os campos](https://www.ensembl.org/info/docs/tools/vep/vep_formats.html#output).

In [ ]:
!./ensembl-vep/vep  \
	-i filtered.vcf.gz  \
	-o filtered.vep.tsv \
  --database --assembly GRCh37 --refseq  \
	--pick --pick_allele --force_overwrite --tab --symbol --check_existing\
  --fields "Location,SYMBOL,Consequence,Feature,Amino_acids,CLIN_SIG" \
	--fasta chr9.fa \
	--individual all 

### Visualizar Tabela Anotada

In [ ]:
!cat filtered.vep.tsv

## Verificar variantes que passaram pelos filtros

O campo `FILTER=PASS` indica as chamadas que satisfizeram os filtros aplicados. A interpretação biológica e clínica exige evidências adicionais.

In [ ]:
!bcftools view -f PASS filtered.vcf.gz